# Week 7 structure_completeness classifier — Colab run

Thin wrapper: every real step lives in a `ml/*.py` script (already self-tested locally with fake data + a tiny random-init model — see `ml/self_test.py`). This notebook just installs deps and runs those scripts in order so the whole pipeline is reproducible from one place.

**Before running:** the data split is already decided and committed to the repo — `ml/data/prepared_samples.json` (150 samples) and `ml/splits/fold_{0..4}.json` + `ml/splits/test.json`. This notebook loads that split, it does **not** re-run `ml/prepare_data.py` / `ml/split_data.py`. If those files are missing after cloning, stop and check the clone/branch rather than regenerating a new split — a new split would no longer match anything already discussed.

**What this notebook does NOT do:** write the final report or a `docs/decision_log.md` entry. That happens after these results exist, as a separate step, so it can reference real numbers instead of guessing at them.

## 0. Get the repo onto this runtime

Pick ONE of the two cells below depending on how you're getting the code here, then delete/skip the other.

In [ ]:
# Option A: clone from your own git remote (fill in your URL — private repos
# need a token in the URL or an SSH key set up in this runtime first).
# !git clone <YOUR_REPO_URL> ai-interview-coach
# %cd ai-interview-coach

In [ ]:
# Option B: repo already uploaded/mounted (e.g. via Google Drive) — just cd into it.
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/ai-interview-coach

In [ ]:
from pathlib import Path

assert Path("ml/data/prepared_samples.json").exists(), (
    "ml/data/prepared_samples.json not found from the current working directory "
    f"({Path.cwd()}) — cd into the repo root first (see the two cells above)."
)
assert Path("ml/splits/test.json").exists(), "ml/splits/ not found — same as above."
print("Repo root looks correct:", Path.cwd())

## 1. Install dependencies

`ml/requirements.txt` is self-contained (doesn't rely on the top-level `requirements.txt`, which this notebook never installs — we only need the `ml/` scripts and the couple of `backend/` modules they import, not the Streamlit app).

In [ ]:
!pip install -q -r ml/requirements.txt

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — Runtime > Change runtime type > GPU, then re-run this cell before training.")

## 2. Groq API key (optional — only for `--llm-baseline` in step 4)

`ml/augment.py`'s back-translation (step 5) now runs a local Helsinki-NLP MarianMT model, not Groq (see docs/decision_log.md's entry on this switch — Groq's free-tier daily token quota is shared with the interview-dialogue feature and got exhausted mid-run). The only thing left in this notebook that still calls Groq is `ml/baselines.py`'s optional `--llm-baseline` zero-shot comparison, which step 4 doesn't run by default.

Skip this cell unless you plan to run `!python ml/baselines.py --llm-baseline` this session — everything else works without it.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY (leave blank + Enter to skip): ")

## 3. Sanity check on THIS runtime before spending any GPU time

Re-runs the same fake-data / tiny-random-model self-test that already passed locally. Cheap (CPU, seconds), but worth confirming again here — a fresh Colab environment can have different library versions than the laptop this was developed on.

In [ ]:
!python ml/self_test.py

## 4. Baselines (majority-class + the app's existing scoring, discretized)

No GPU needed — this also already ran locally (see `ml/results/baselines/baseline_results.json` if it came over with the clone). Re-running here just confirms the same numbers reproduce in this environment; skip this cell if you just want to get to training.

In [ ]:
!python ml/baselines.py

## 5. Data augmentation (back-translation, train folds only)

Runs a local Helsinki-NLP MarianMT model (opus-mt-zh-en / opus-mt-en-zh) on this runtime's GPU if available (falls back to CPU otherwise — check step 1's CUDA cell). Switched over from calling Groq: that approach hit Groq's free-tier daily token quota mid-run (shared with the interview-dialogue feature's use of the same model), which no retry/backoff tuning fixes — see docs/decision_log.md for the full writeup. The whole fold's samples are translated in batches (not one call per item) for GPU throughput.

**Note on `ml/data/augmented/fold_0_train_augmented.json`**: that file was produced by the *old* Groq-based translator (102/102, spot-checked in `ml/augment_review.md`), not MarianMT. It'll be skipped by the idempotent-fold check below like any other already-done fold — pass `--force` if you want fold 0 redone with the same (local) translator as the rest, for consistency across folds. `ml/augment_review.md`'s "no structure-rewriting found" conclusion was checked against the Groq output specifically and hasn't been re-verified against MarianMT output; spot-check a few MarianMT pairs by eye if augmented-vs-baseline metrics in step 8 look off.

**Locally self-tested**: `ml/self_test.py` section 4 covers the augment/leakage logic with a fake `translate_fn` (no model download, no network calls beyond what pip already fetched). Writes `ml/data/augmented/fold_{0..4}_train_augmented.json` (running without `--fold` does all 5). **Idempotent per fold**: if a fold's output file already exists and is non-empty, it's skipped (printed, not silent) rather than re-translated. Pass `--force` to redo a fold anyway. Also writes `fold_{i}_train_augmented.skip_report.json` per fold (even when nothing was skipped) recording any sample that failed translation (OOM even after retrying at a smaller sub-batch, or the model failing to load) — check it for `skipped_count > 0` after this runs; a nonzero count means that fold's augmented training data is short those samples and the fold should be re-run with `--force` once the cause is fixed.

In [ ]:
!python ml/augment.py

## 6. Train xlm-roberta-base (primary backbone, all 5 folds)

**Needs GPU** (Runtime > Change runtime type > GPU, then re-check step 1's CUDA cell) — this is the actual fine-tuning run; it will technically run on CPU but is impractically slow there. **Locally self-tested**: `ml/self_test.py` section 7 already ran this exact code path (model building, layer freezing, discriminative-LR optimizer, `Trainer` + label smoothing + early stopping) end-to-end on CPU, using a tiny random-initialized xlm-roberta-base config — that confirms the wiring, it says nothing about real accuracy (fake data, fake weights there). This cell is the first time the real pretrained weights run against the real 150-sample dataset.

Defaults already match the task spec (freeze embeddings + bottom 7 layers, discriminative LR, dropout 0.4, label smoothing 0.1, up to 25 epochs with early stopping patience 4, batch size 16, max_length from `ml/data_summary.md`'s length distribution). Override any of them with flags — see `!python ml/train.py --help`.

In [ ]:
!python ml/train.py --model xlm-roberta-base

## 7. If step 6 printed an overfitting warning: try the smaller model

**Needs GPU**, same as step 6. **Locally self-tested** the same way — `ml/self_test.py` section 7's tiny-random-config smoke test covers both `xlm-roberta-base` and `distilbert-base-multilingual-cased`, so this backbone's wiring is equally verified, just not yet run with real weights.

`ml/train.py` prints `WARNING: Overfitting signal: ...` at the end if mean train macro-F1 exceeds mean val macro-F1 by more than 0.15. If that happened, run the smaller `distilbert-base-multilingual-cased` backbone for comparison. If it didn't happen, skip this cell.

In [ ]:
!python ml/train.py --model distilbert-base-multilingual-cased

## 8. Augmented training run + before/after comparison

**Needs GPU**, same as step 6 — another full 5-fold training run, just with augmented training data mixed in; the training code path itself is identical to step 6 (already self-tested there), only the training set is bigger. Only meaningful if step 5 actually ran for **all 5 folds** (not just the fold-0 spot-check from the local run — check `ml/data/augmented/` has `fold_0` through `fold_4` before running this). Writes to a separate `ml/results/train/xlm-roberta-base__augmented/` path — `--augment` never overwrites the plain run from step 6, so both are available to compare.

In [ ]:
!python ml/train.py --model xlm-roberta-base --augment

In [ ]:
import json

import ml.augment as augment

before = json.load(open("ml/results/train/xlm-roberta-base/summary.json", encoding="utf-8"))
after = json.load(open("ml/results/train/xlm-roberta-base__augmented/summary.json", encoding="utf-8"))
print(augment.compare_metrics(
    before, after,
    keys=("mean_val_macro_f1", "mean_val_qwk", "mean_val_within1_accuracy"),
))

## 8b. If step 8's before/after diff looks non-trivial (either direction): length-artifact cross-check

**No GPU needed** — reuses the two runs' already-saved `oof_predictions.json` files, no model inference. **Locally self-tested**: `ml/self_test.py` section 6 covers `compare_length_bias()` with a fabricated case engineered so only the longest-answer samples "improve," and confirms the tool actually flags it (clearly negative length-vs-delta correlation, long tertile far more negative than short tertile).

Why this step exists: back-translation makes AUGMENTED TRAINING text run ~23% longer on average (`ml/augment_review.md`) — a translation side effect, not a real structure change. The val/test sets are never augmented, so if the augmented model quietly learned "longer answer -> higher band" as a shortcut instead of a real structural signal, it would show up here as: errors improving disproportionately on already-long true answers. Skip this cell if step 8's mean_val_macro_f1/qwk barely moved — it's specifically for explaining a *surprising* jump (up or down), not a routine check on every run.

In [ ]:
!python ml/error_analysis.py --model xlm-roberta-base --compare-augmentation

## 9. Learning curve (time permitting — lower priority than the steps above)

**Needs GPU**, same as step 6 — it's 4 more training runs (25/50/75/100% of one fold's data), using the exact same `train_one_fold()` under the hood (already self-tested there). The part unique to this script — nested stratified subsampling by fraction — has its own coverage in `ml/self_test.py` section 5 (checks the subsets nest correctly and are reproducible across runs), tested with fake data and no actual training involved.

Defaults to a single fold (fold 0) to keep the ~4x extra training cost bounded; pass `--folds 0 1 2 3 4` for a fuller (and ~4x slower again) picture.

In [ ]:
!python ml/learning_curve.py --model xlm-roberta-base

## 10. Error analysis

**No GPU needed** — pure post-hoc analysis over already-saved predictions, no model inference happens here. **Locally self-tested**: `ml/self_test.py` section 6 covers `summarize_errors()`/`render_report()` with fabricated predictions (including checking the single-annotator caveat text is always present), and separately covers `compare_length_bias()`/`length_error_correlation()` (the length-artifact cross-check below) with a fabricated "only long answers improved" case, confirming it actually gets flagged.

Needs `ml/results/train/<model>/oof_predictions.json`, which step 6 (or 7/8) only writes once *all 5* folds have completed. Reads whichever `--model` you point it at.

In [ ]:
!python ml/error_analysis.py --model xlm-roberta-base

## 11. Final model — fit once on train+val, evaluate once on the held-out test set

**Needs GPU.** This is the report-facing run, done *after* steps 6-8 have already picked the model/recipe (Week 7 decision: `distilbert-base-multilingual-cased`, no augmentation — see `docs/decision_log.md`). `--final` trains once on the full ~128-sample train+val pool (all non-test ids) and evaluates **once**, after training is fully finished, on the ~22-sample held-out `ml/splits/test.json` set — that set is never used for training or for any epoch/hyperparameter decision (see `train_final_model()`'s docstring in `ml/train.py` for the isolation guarantee, enforced by a `ValueError` if the two id sets ever overlap).

There's no validation split left in this mode, so early stopping isn't possible — `--final-epochs` is a fixed budget instead. If step 7's `summary.json` for this model/augment combination has a `mean_best_epoch` field (added alongside `--final`), pass that value explicitly; otherwise the default (`ml/train.py`'s `DEFAULT_FINAL_EPOCHS`, currently 14 — the middle of the ~10-18 range distilbert's CV folds converged at) is used automatically.

In [ ]:
import json

try:
    mean_best_epoch = json.load(open("ml/results/train/distilbert-base-multilingual-cased/summary.json", encoding="utf-8")).get("mean_best_epoch")
except FileNotFoundError:
    mean_best_epoch = None

final_epochs_flag = f"--final-epochs {round(mean_best_epoch)}" if mean_best_epoch else ""
print("Using --final-epochs", round(mean_best_epoch) if mean_best_epoch else "(unset -> DEFAULT_FINAL_EPOCHS)")
!python ml/train.py --model distilbert-base-multilingual-cased --final {final_epochs_flag}

## 12. Pull results back off this (ephemeral) runtime

Everything lands under `ml/results/`. Zip it up so it survives the runtime being recycled — download it, or copy to the mounted Drive if using Option B from step 0.

In [ ]:
!zip -qr ml_results.zip ml/results ml/data/augmented
print("Wrote ml_results.zip — download it (Files pane) or copy it to Drive before this runtime recycles.")